# P02 第二部分：数据清洗与存储

含完整清洗、宽长表转换、合并及 Parquet/SQLite 演示代码。

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import time

import numpy as np
import pandas as pd

# ---------- 项目根目录 ----------
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    ROOT = Path(r"G:/ganlijie/ds_homework/dshw-p01")

START_DATE = "20200101"
END_DATE = datetime.now().strftime("%Y%m%d")

STOCK_LIST = [
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "601633", "name": "长城汽车", "industry": "汽车"},
    {"code": "000002", "name": "万科A", "industry": "房地产"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "name": "五粮液", "industry": "白酒"},
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    {"code": "600941", "name": "中国移动", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

DIRS = {
    "stock": ROOT / "data" / "stock",
    "index": ROOT / "data" / "index",
    "macro": ROOT / "data" / "macro",
    "finance": ROOT / "data" / "finance",
    "clean": ROOT / "data" / "clean",
    "combined": ROOT / "data" / "combined",
    "output": ROOT / "output",
}
LOG_FILE = ROOT / "download_log.txt"
COMBINED_CSV = DIRS["combined"] / "combined_data.csv"
STOCK_CLEAN_CSV = DIRS["clean"] / "stock_clean.csv"
STOCK_CLEAN_PARQUET = DIRS["clean"] / "stock_clean.parquet"
FIN_DB = DIRS["combined"] / "fin_data.db"
RF_DAILY = 0.02 / 252
TRADING_DAYS = 252
FINANCE_INDICATORS = ["净资产收益率(ROE)", "销售净利率"]

def setup_directories():
    for p in DIRS.values():
        p.mkdir(parents=True, exist_ok=True)

def log_download(status, tag, detail=""):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {status:<7} {tag}"
    if detail:
        line += f"  {detail}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def stock_csv_path(code):
    return DIRS["stock"] / f"stock_{code}.csv"

def index_csv_path(code):
    return DIRS["index"] / f"index_{code}.csv"

setup_directories()

try:
    from IPython.display import display
except ImportError:
    display = print

print("项目根目录:", ROOT.resolve())

项目根目录: G:\ganlijie\ds_homework\dshw-p01


In [ ]:
def clean_single_stock(df):
    stats = {}
    n0 = len(df)
    stats["missing_before"] = df.isna().sum().to_dict()
    price_cols = ["open", "close", "high", "low"]
    for c in price_cols:
        if df[c].dtype == object:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df[price_cols] = df[price_cols].ffill()
    df[["volume", "amount"]] = df[["volume", "amount"]].fillna(0)
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    n_dup = int(df.index.duplicated(keep="first").sum())
    stats["duplicates_removed"] = n_dup
    df = df[~df.index.duplicated(keep="first")]
    df["return"] = np.log(df["close"] / df["close"].shift(1))
    df["is_extreme"] = df["return"].abs() > 0.20
    stats["rows_before"] = n0
    stats["rows_after"] = len(df)
    return df, stats

def clean_all_stocks():
    frames, all_stats = [], []
    for item in STOCK_LIST:
        code = item["code"]
        raw = pd.read_csv(stock_csv_path(code), parse_dates=["date"])
        raw["code"] = str(code).zfill(6)
        cleaned, st = clean_single_stock(raw)
        cleaned = cleaned.reset_index()
        cleaned["code"] = str(code).zfill(6)
        frames.append(cleaned)
        st["code"] = str(code).zfill(6)
        all_stats.append(st)
    long_df = pd.concat(frames, ignore_index=True)
    long_df["code"] = long_df["code"].astype(str).str.zfill(6)
    long_df.to_csv(STOCK_CLEAN_CSV, index=False, encoding="utf-8-sig")
    long_df.to_parquet(STOCK_CLEAN_PARQUET, index=False)
    return long_df, pd.DataFrame(all_stats)

def wide_close_table(long_df):
    return long_df.pivot(index="date", columns="code", values="close")

def melt_to_long(wide):
    melted = wide.reset_index().melt(id_vars="date", var_name="code", value_name="close")
    melted["date"] = pd.to_datetime(melted["date"])
    melted["code"] = melted["code"].astype(str).str.zfill(6)
    return melted

def load_index(code):
    df = pd.read_csv(index_csv_path(code), parse_dates=["date"])
    return df.rename(columns={"close": f"index_{code}_close"})[["date", f"index_{code}_close"]]

def load_macro():
    frames = [pd.read_csv(p, parse_dates=["date"]) for p in DIRS["macro"].glob("macro_*.csv")]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def merge_daily_with_index(long_df):
    m = long_df.merge(load_index("000300"), on="date", how="left")
    return m.merge(load_index("399006"), on="date", how="left")

def merge_macro_to_daily(daily, macro):
    daily = daily.copy()
    daily["month"] = pd.to_datetime(daily["date"]).dt.to_period("M").astype(str)
    macro = macro.copy()
    macro["month"] = pd.to_datetime(macro["date"]).dt.to_period("M").astype(str)
    macro_wide = macro.pivot(index="month", columns="indicator", values="value").reset_index()
    return daily.merge(macro_wide, on="month", how="left")

def build_sqlite_db(long_df, macro):
    import sqlite3
    if FIN_DB.exists():
        FIN_DB.unlink()
    conn = sqlite3.connect(FIN_DB)
    price = long_df[["code", "date", "close", "volume"]].copy()
    price["date"] = price["date"].astype(str)
    price.to_sql("stock_price", conn, if_exists="replace", index=False)
    if not macro.empty:
        m = macro.copy()
        m["date"] = pd.to_datetime(m["date"]).dt.strftime("%Y-%m-%d")
        m.to_sql("macro_data", conn, if_exists="replace", index=False)
    pd.DataFrame(STOCK_LIST)[["code", "name", "industry"]].to_sql(
        "stock_info", conn, if_exists="replace", index=False
    )
    conn.close()

print("清洗函数已定义")

清洗函数已定义


## 3.1 单表清洗

In [ ]:
code0 = STOCK_LIST[0]['code']
raw = pd.read_csv(stock_csv_path(code0), parse_dates=['date'])
print('清洗前缺失:\n', raw.isna().sum())
long_df, stats_df = clean_all_stocks()
display(stats_df[['code','rows_before','rows_after','duplicates_removed']])
display(long_df.head())

清洗前缺失:
 date      0
open      0
close     0
high      0
low       0
volume    0
amount    0
code      0
dtype: int64


,code,rows_before,rows_after,duplicates_removed
0,601398,1541,1541,0
1,600036,1541,1541,0
2,002594,1541,1541,0
3,601633,1541,1541,0
4,000002,1541,1541,0
5,600519,1541,1541,0
6,000858,1541,1541,0
7,601857,1541,1541,0
8,600941,1054,1054,0
9,002352,1538,1538,0


,date,open,close,high,low,volume,amount,code,return,is_extreme
0,2020-01-02,8.73,8.78,8.85,8.72,2349494,1.404443e+09,601398,NaN,False
1,2020-01-03,8.78,8.81,8.84,8.77,1522130,9.119516e+08,601398,0.003411,False
2,2020-01-06,8.77,8.78,8.87,8.76,2265097,1.359917e+09,601398,-0.003411,False
3,2020-01-07,8.80,8.83,8.86,8.80,1168044,7.016158e+08,601398,0.005679,False
4,2020-01-08,8.77,8.72,8.78,8.71,1585591,9.405527e+08,601398,-0.012536,False


**说明**：价格 `ffill` 处理停牌；涨跌幅超 ±20% 标注 `is_extreme`，不删除。

## 3.2 宽表与长表

In [ ]:
wide = wide_close_table(long_df)
melted = melt_to_long(wide)
print('宽表', wide.shape, '→ 长表', melted.shape)
display(melted.head())

宽表 (1541, 10) → 长表 (15410, 3)


,date,code,close
0,2020-01-02,000002,4919.41
1,2020-01-03,000002,4852.65
2,2020-01-06,000002,4781.96
3,2020-01-07,000002,4814.68
4,2020-01-08,000002,4804.21


**宽表**适合相关矩阵；**长表**适合面板回归。

## 3.3 多表合并

In [ ]:
n0 = len(long_df)
merged_idx = merge_daily_with_index(long_df)
macro = load_macro()
combined = merge_macro_to_daily(merged_idx, macro)
combined.to_csv(COMBINED_CSV, index=False, encoding='utf-8-sig')
build_sqlite_db(long_df, macro)
print('合并指数后行数', len(merged_idx), '| 合并宏观后', len(combined))
display(combined.head())

合并指数后行数 14920 | 合并宏观后 14920


,date,open,close,high,low,volume,amount,code,return,is_extreme,index_000300_close,index_399006_close,month,cpi,m2
0,2020-01-02,8.73,8.78,8.85,8.72,2349494,1.404443e+09,601398,NaN,False,4152.241,1832.742,2020-01,0.0,8.4
1,2020-01-03,8.78,8.81,8.84,8.77,1522130,9.119516e+08,601398,0.003411,False,4144.965,1836.012,2020-01,0.0,8.4
2,2020-01-06,8.77,8.78,8.87,8.76,2265097,1.359917e+09,601398,-0.003411,False,4129.295,1859.921,2020-01,0.0,8.4
3,2020-01-07,8.80,8.83,8.86,8.80,1168044,7.016158e+08,601398,0.005679,False,4160.227,1893.214,2020-01,0.0,8.4
4,2020-01-08,8.77,8.72,8.78,8.71,1585591,9.405527e+08,601398,-0.012536,False,4112.317,1862.699,2020-01,0.0,8.4


In [ ]:
import pyarrow.parquet as pq
schema = pq.read_schema(str(STOCK_CLEAN_PARQUET))
print(schema)
t0 = time.time(); pd.read_csv(STOCK_CLEAN_CSV)
print(f"CSV     {time.time()-t0:.3f}s  {os.path.getsize(STOCK_CLEAN_CSV)/1024:.1f} KB")
t0 = time.time(); pd.read_parquet(STOCK_CLEAN_PARQUET)
print(f"Parquet {time.time()-t0:.3f}s  {os.path.getsize(STOCK_CLEAN_PARQUET)/1024:.1f} KB")

date: timestamp[us]
open: double
close: double
high: double
low: double
volume: int64
amount: double
code: large_string
return: double
is_extreme: bool
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1200
CSV     0.015s  1358.5 KB
Parquet 0.020s  683.2 KB


数据规模较小时差异不明显；数据量增大后 Parquet 列式优势更显著。

## 方式 C：SQLite 查询

In [ ]:
import sqlite3
conn = sqlite3.connect(FIN_DB)
q1 = '''SELECT p.date, p.code, p.close, m.value AS cpi
FROM stock_price p LEFT JOIN macro_data m
  ON substr(p.date,1,7)=substr(m.date,1,7) AND m.indicator='cpi' LIMIT 5'''
print('查询1 行情+CPI:'); display(pd.read_sql_query(q1, conn))
q2 = "SELECT date, code, volume FROM stock_price ORDER BY volume DESC LIMIT 3"
print('查询2 成交量前三:'); display(pd.read_sql_query(q2, conn))
q3 = '''SELECT i.industry, AVG(p.close) avg_close
FROM stock_price p JOIN stock_info i ON p.code=i.code
GROUP BY i.industry ORDER BY avg_close DESC'''
print('查询3 行业均价:'); display(pd.read_sql_query(q3, conn))
conn.close()

查询1 行情+CPI:


,date,code,close,cpi
0,2020-01-02,601398,8.78,0.0
1,2020-01-03,601398,8.81,0.0
2,2020-01-06,601398,8.78,0.0
3,2020-01-07,601398,8.83,0.0
4,2020-01-08,601398,8.72,0.0


查询2 成交量前三:


,date,code,volume
0,2024-10-08,601398,16882262
1,2024-09-30,601398,16161742
2,2020-11-30,601398,12456636


查询3 行业均价:


,industry,avg_close
0,白酒,6042.965681
1,房地产,3106.094322
2,汽车,168.279533
3,物流,164.593628
4,通讯,101.898207
5,银行,99.420243
6,能源,10.422875
